#Setup API Keys

In [5]:
from google.colab import userdata
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
GROQ_API_KEY = userdata.get('GROQ_API_KEY')

#Install Libraries

In [6]:
!pip install -q langchain==1.0.5 langchain-groq==1.0.0 langchain-google-genai==3.0.1 langchain-community==0.4.1 chromadb==1.3.4 pypdf

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.8/93.8 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 58.1/58.1 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 42.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.8/20.8 MB 66.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 336.3/336.3 kB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 66.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 10.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 72.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 52.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 515.1/515.1 kB 33.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 160.9/160

In [7]:
!pip show langchain langchain-groq langchain-google-genai langchain-community chromadb pypdf

Name: langchain
Version: 1.0.5
Summary: Building applications with LLMs through composability
Home-page: https://docs.langchain.com/
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: langchain-core, langgraph, pydantic
Required-by: 
---
Name: langchain-groq
Version: 1.0.0
Summary: An integration package connecting Groq and LangChain
Home-page: https://docs.langchain.com/oss/python/integrations/providers/groq
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: groq, langchain-core
Required-by: 
---
Name: langchain-google-genai
Version: 3.0.1
Summary: An integration package connecting Google's genai package and LangChain
Home-page: 
Author: 
Author-email: 
License: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: filetype, google-ai-generativelanguage, langchain-core, pydantic
Required-by: 
---
Name: langchain-community
Version: 0.4.1
Summary: Community contributed LangChain integra

#load the document

In [9]:
from langchain_community.document_loaders import PyPDFLoader

file_path = "/content/scholarship_info.pdf"
loader = PyPDFLoader(file_path)
doc = loader.load()


In [10]:
doc

[Document(metadata={'producer': 'Microsoft® Word 2019', 'creator': 'Microsoft® Word 2019', 'creationdate': '2025-08-07T19:16:29+05:30', 'author': 'Preethesh Poojary', 'moddate': '2025-08-07T19:16:29+05:30', 'source': '/content/scholarship_info.pdf', 'total_pages': 1, 'page': 0, 'page_label': '1'}, page_content='Title: Scholarship Information 2025 \n \n1. Eligibility: \n- Open to students in India pursuing undergraduate degrees. \n- Annual family income must be below ₹6,00,000. \n- Minimum 60% marks in the last qualifying exam. \n \n2. Documents Required: \n- Income certificate \n- Aadhaar card \n- Bank passbook \n- Marksheet \n \n3. Deadline: October 15, 2025 \n \n4. Benefits: \n- ₹10,000 per semester for tuition \n- Book allowance of ₹3,000 per year \n \n5. How to Apply: \nVisit https://scholarships.gov.in and register under the NSP portal.')]

#split the document

In [11]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
     chunk_size= 500,
     chunk_overlap=200
)

splits = splitter.split_documents(doc)

print(f"Split into {len(splits)}")

Split into 2


#Create emdeddings and store them in chroma

In [14]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_classic.vectorstores import Chroma

embeddings = GoogleGenerativeAIEmbeddings(
    model = "models/gemini-embedding-001",
    google_api_key = GEMINI_API_KEY
)

vector_store = Chroma(
    embedding_function= embeddings,
    persist_directory="rag_db",
    collection_name="scholarship_info"

)

vector_store.add_documents(splits)

['b94ab90a-8f6b-48a8-bf28-436572146442',
 'b1c06f6e-7b53-48ad-a879-45e25a9bf521']

#Create Retriever

In [20]:
retriever = vector_store.as_retriever(search_kwargs={"k" : 2})

#Initialize the Groq LLm

In [18]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model = "openai/gpt-oss-20b",
    api_key = GROQ_API_KEY,
    temperature= 0.3,
    max_tokens= 200
)



#Create RAG Chain

In [22]:
from langchain_classic.chains import RetrievalQA

rag_chain = RetrievalQA.from_chain_type(
    llm = llm,
    retriever = retriever
)


#Ask Questions

In [24]:
query = "Who is eligible for the Scholarship?"
response = rag_chain.invoke({"query": query})
print(response["result"])

**Eligibility for the Scholarship**

- **Student status:** Must be a student in India who is currently pursuing an undergraduate degree.  
- **Family income:** Annual family income must be **below ₹6,00,000**.  
- **Academic performance:** Must have scored a minimum of **60 %** in the last qualifying examination (e.g., 12th grade, diploma, etc.).  

Only students who meet all three of these criteria are eligible to apply.
